In [7]:
import json
import pickle
import os
import threading
import pandas as pd
from flask import Flask, request, jsonify
import requests
import datetime

# Set a fixed random-state value for reproducibility where required.
RANDOM_STATE = 42

In [2]:
# Upload Pretrained Model for API
from google.colab import files

# Ask the user to upload the trained Pickle model.
print("Please upload the trained model file: yield_model_pipeline.pkl")
uploaded_model = files.upload()

# Get the uploaded model filename.
model_file = next(iter(uploaded_model))

# Open the uploaded model file in binary-read mode.
with open(model_file, 'rb') as f:
    bundle = pickle.load(f)

# Extract the trained preprocessing and prediction pipeline.
model = bundle['pipeline']

# Extract the numerical features expected by the model.
num_feats = bundle['num_feats']

# Extract the categorical features expected by the model.
cat_feats = bundle['cat_feats']

# Combine all required input fields into one list.
required_fields = num_feats + cat_feats

# Display confirmation.
print("\nModel uploaded and loaded successfully.")
print("Model file:", model_file)

# Display the fields that must be supplied in every prediction request.
print("Endpoint will require these fields in every request:")
print(required_fields)

Please upload the trained model file: yield_model_pipeline.pkl


Saving yield_model_pipeline.pkl to yield_model_pipeline.pkl

Model uploaded and loaded successfully.
Model file: yield_model_pipeline.pkl
Endpoint will require these fields in every request:
['urban', 'family_income', 'first_gen', 'parent_grad', 'cutoff_12th', 'entrance_score', 'tuition', 'distance_km', 'competing_offers', 'merit_aid_pct', 'need_aid_pct', 'total_aid_pct', 'aid_amount', 'net_price', 'district', 'category', 'college_tier']


In [10]:
# --- Where prediction outputs get saved on the Colab filesystem ---
# Every call to /predict will also write its JSON response to a file here,
# in addition to returning it over HTTP, so the results persist as files
# you can browse (Colab's left-hand Files pane) or download afterwards.
# Saved directly under Colab's default /content/ directory -- no extra subfolder.
OUTPUT_DIR = '/content'

LATEST_OUTPUT_PATH = os.path.join(
    OUTPUT_DIR,
    'latest_prediction.json'
)

print('Prediction JSON files will be saved under:', OUTPUT_DIR)

Prediction JSON files will be saved under: /content


In [3]:
# Create the Flask application object.
app = Flask(__name__)

# Define the fields that must contain numeric values in an incoming request.
NUMERIC_FIELDS = {"urban", "family_income", "first_gen", "parent_grad", "cutoff_12th", "entrance_score",
                   "tuition", "distance_km", "competing_offers", "merit_aid_pct", "need_aid_pct",
                   "total_aid_pct", "aid_amount", "net_price"}

# Validate the applicant data received by the API.
def validate_payload(payload):
    # Make sure the request body is a JSON object/dictionary.
    if not isinstance(payload, dict):
        return "Request body must be a JSON object of applicant features."

    # Identify any required fields that are missing from the request.
    missing = [f for f in required_fields if f not in payload]
    if missing:
        return f"Missing required fields: {missing}"

    # Check that all numeric fields contain int or float values.
    bad_types = [f for f in NUMERIC_FIELDS if f in payload and not isinstance(payload[f], (int, float))]
    if bad_types:
        return f"These fields must be numeric: {bad_types}"

    # Return None when the payload passes all validation checks.
    return None


def save_prediction_output(payload, response_body):
    # Write one timestamped JSON file per successful prediction, plus overwrite
    # 'latest_prediction.json' so there's always a fixed, known path to read from.
    record = {
        "timestamp": datetime.datetime.now().isoformat(),
        "input": payload,
        "output": response_body,
    }
    filename = f"prediction_{datetime.datetime.now().strftime('%Y%m%dT%H%M%S%f')}.json"
    filepath = os.path.join(OUTPUT_DIR, filename)

    with open(filepath, 'w') as f:
        json.dump(record, f, indent=2)
    with open(LATEST_OUTPUT_PATH, 'w') as f:
        json.dump(record, f, indent=2)

    return filepath


# Define the health-check endpoint using the HTTP GET method.
@app.route('/health', methods=['GET'])
def health():
    # Return a simple status message and confirm that the model is loaded.
    return jsonify({"status": "ok", "model_loaded": model is not None}), 200


# Define the prediction endpoint using the HTTP POST method.
@app.route('/predict', methods=['POST'])
def predict():
    # Read the incoming request body as JSON.
    payload = request.get_json(silent=True)

    # Validate the received applicant information.
    error = validate_payload(payload)

    # Return HTTP 400 when the request is invalid. This is expected, correct
    # behavior for a malformed request (missing fields, wrong types, or a
    # non-object body) -- it is not a server error, and the three 400s you'll
    # see in the validation-test cell below are deliberate: they're testing
    # that this check actually rejects bad input rather than crashing on it.
    if error:
        return jsonify({"error": error}), 400

    try:
        # Convert the applicant dictionary into a one-row DataFrame.
        row = pd.DataFrame([payload])[required_fields]

        # Use the pretrained pipeline to calculate the enrollment probability.
        proba = float(model.predict_proba(row)[:, 1][0])

        # Convert the probability into a binary enrollment prediction.
        predicted_enrolled = proba >= 0.5

        response_body = {
            "yield_probability": proba,
            "predicted_enrolled": predicted_enrolled,
        }

        # Persist this prediction to the Colab filesystem as a JSON file.
        saved_path = save_prediction_output(payload, response_body)
        response_body["saved_to"] = saved_path

        # Return the prediction as a JSON response with HTTP 200.
        return jsonify(response_body), 200

    except Exception as e:
        # Return HTTP 500 if an unexpected inference error occurs.
        return jsonify({"error": str(e)}), 500


In [4]:
# Define a function that starts the Flask API server.
def run_app():
    # Listen on all network interfaces using port 5000.
    # Disable debug mode and the reloader to avoid duplicate server processes in Colab.
    app.run(host='0.0.0.0', port=5000, debug=False, use_reloader=False)


def is_server_already_running():
    # Re-running this cell (a common Colab habit) used to try to bind port 5000
    # a second time and print "Address already in use" -- harmless, since the
    # first server instance was still serving, but confusing. This check makes
    # a rerun a no-op instead of a misleading error message.
    try:
        r = requests.get('http://127.0.0.1:5000/health', timeout=1)
        return r.status_code == 200
    except Exception:
        return False


if is_server_already_running():
    print('Server already running on http://127.0.0.1:5000 -- reusing the existing instance.')
else:
    # Start Flask in a background thread so the notebook remains usable.
    server_thread = threading.Thread(target=run_app, daemon=True)
    server_thread.start()

    # Import time so the notebook can briefly wait for the server to start.
    import time
    time.sleep(2)  # Give Flask a moment to bind to port 5000.

    # Display a message confirming that the local server has started.
    print('Server started on http://127.0.0.1:5000')


 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://172.28.0.12:5000
INFO:werkzeug:Press CTRL+C to quit


Server started on http://127.0.0.1:5000


In [8]:
# Send a GET request to the health endpoint to verify that the API is running.
r = requests.get('http://127.0.0.1:5000/health')

# Display the HTTP status code and JSON response returned by Flask.
print('GET /health ->', r.status_code, r.json())

INFO:werkzeug:127.0.0.1 - - [18/Sep/2026 05:49:32] "GET /health HTTP/1.1" 200 -


GET /health -> 200 {'model_loaded': True, 'status': 'ok'}


In [11]:
# Define a sample applicant whose features match the model's required input schema.
sample_applicant = {
    'district': 'Madurai', 'category': 'MBC', 'urban': 0, 'family_income': 210000,
    'first_gen': 1, 'parent_grad': 0, 'cutoff_12th': 79.0, 'entrance_score': 128.0,
    'college_tier': 'Tier-2 (Affiliated)', 'tuition': 110000, 'distance_km': 35.0,
    'competing_offers': 1, 'merit_aid_pct': 0.20, 'need_aid_pct': 0.35,
    'total_aid_pct': 0.28, 'aid_amount': 30800, 'net_price': 79200,
}

# Send the applicant data to the /predict endpoint using a POST request.
r = requests.post('http://127.0.0.1:5000/predict', json=sample_applicant)

# Display the response status code and formatted prediction result.
print('POST /predict ->', r.status_code)
print(json.dumps(r.json(), indent=2))


INFO:werkzeug:127.0.0.1 - - [18/Sep/2026 05:50:34] "POST /predict HTTP/1.1" 200 -


POST /predict -> 200
{
  "predicted_enrolled": false,
  "saved_to": "/content/prediction_20260918T055034166456.json",
  "yield_probability": 0.3799366710209084
}


In [12]:
# Prove the JSON output actually landed on disk at the Colab file path above,
# by listing the saved files and reading the most recent one back.
import glob

saved_files = sorted(glob.glob(os.path.join(OUTPUT_DIR, 'prediction_*.json')))
print(f"{len(saved_files)} prediction file(s) saved in {OUTPUT_DIR}")
print('Most recent file:', saved_files[-1] if saved_files else None)
print()

with open(LATEST_OUTPUT_PATH) as f:
    print('Contents of', LATEST_OUTPUT_PATH, ':')
    print(json.dumps(json.load(f), indent=2))


1 prediction file(s) saved in /content
Most recent file: /content/prediction_20260918T055034166456.json

Contents of /content/latest_prediction.json :
{
  "timestamp": "2026-09-18T05:50:34.166438",
  "input": {
    "district": "Madurai",
    "category": "MBC",
    "urban": 0,
    "family_income": 210000,
    "first_gen": 1,
    "parent_grad": 0,
    "cutoff_12th": 79.0,
    "entrance_score": 128.0,
    "college_tier": "Tier-2 (Affiliated)",
    "tuition": 110000,
    "distance_km": 35.0,
    "competing_offers": 1,
    "merit_aid_pct": 0.2,
    "need_aid_pct": 0.35,
    "total_aid_pct": 0.28,
    "aid_amount": 30800,
    "net_price": 79200
  },
  "output": {
    "yield_probability": 0.3799366710209084,
    "predicted_enrolled": false
  }
}


In [13]:
# Test 1: Send an incomplete applicant record to verify missing-field validation.
incomplete = {'district': 'Chennai', 'family_income': 300000}
r = requests.post('http://127.0.0.1:5000/predict', json=incomplete)
print('Missing fields ->', r.status_code, r.json())
print()

# Test 2: Replace a numeric field with text to verify type validation.
bad_type = dict(sample_applicant)
bad_type['family_income'] = 'a lot'
r = requests.post('http://127.0.0.1:5000/predict', json=bad_type)
print('Bad type ->', r.status_code, r.json())
print()

# Test 3: Send a JSON list instead of an applicant object.
# This verifies that the API rejects a request body with the wrong structure.
r = requests.post('http://127.0.0.1:5000/predict', json=[1, 2, 3])
print('Non-object body ->', r.status_code, r.json())

INFO:werkzeug:127.0.0.1 - - [18/Sep/2026 05:50:36] "POST /predict HTTP/1.1" 400 -
INFO:werkzeug:127.0.0.1 - - [18/Sep/2026 05:50:36] "POST /predict HTTP/1.1" 400 -
INFO:werkzeug:127.0.0.1 - - [18/Sep/2026 05:50:36] "POST /predict HTTP/1.1" 400 -


Missing fields -> 400 {'error': "Missing required fields: ['urban', 'first_gen', 'parent_grad', 'cutoff_12th', 'entrance_score', 'tuition', 'distance_km', 'competing_offers', 'merit_aid_pct', 'need_aid_pct', 'total_aid_pct', 'aid_amount', 'net_price', 'category', 'college_tier']"}

Bad type -> 400 {'error': "These fields must be numeric: ['family_income']"}

Non-object body -> 400 {'error': 'Request body must be a JSON object of applicant features.'}


In [14]:
# Create a list of applicants to demonstrate repeated API requests.
applicant_batch = [
    {'district': 'Chennai', 'category': 'OC', 'urban': 1, 'family_income': 950000, 'first_gen': 0,
     'parent_grad': 1, 'cutoff_12th': 91.0, 'entrance_score': 160.0, 'college_tier': 'Tier-1 (Autonomous)',
     'tuition': 180000, 'distance_km': 8.0, 'competing_offers': 3, 'merit_aid_pct': 0.10, 'need_aid_pct': 0.05,
     'total_aid_pct': 0.08, 'aid_amount': 14400, 'net_price': 165600},
    {'district': 'Villupuram', 'category': 'SC', 'urban': 0, 'family_income': 95000, 'first_gen': 1,
     'parent_grad': 0, 'cutoff_12th': 71.0, 'entrance_score': 100.0, 'college_tier': 'Tier-3 (Self-financing)',
     'tuition': 85000, 'distance_km': 70.0, 'competing_offers': 0, 'merit_aid_pct': 0.15, 'need_aid_pct': 0.55,
     'total_aid_pct': 0.35, 'aid_amount': 29800, 'net_price': 55200},
]

# Send each applicant to the same /predict endpoint one at a time.
for applicant in applicant_batch:
    # Submit the applicant data as JSON using an HTTP POST request.
    r = requests.post('http://127.0.0.1:5000/predict', json=applicant)

    # Read the JSON response returned by the API.
    result = r.json()

    # Display the applicant's district and prediction results.
    print(f"{applicant['district']:12s} yield_probability={result['yield_probability']}  predicted_enrolled={result['predicted_enrolled']}")

INFO:werkzeug:127.0.0.1 - - [18/Sep/2026 05:50:37] "POST /predict HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [18/Sep/2026 05:50:37] "POST /predict HTTP/1.1" 200 -


Chennai      yield_probability=0.6919352580242537  predicted_enrolled=True
Villupuram   yield_probability=0.25713273816913335  predicted_enrolled=False
